# DCGAN on SVHN

SVHN cropped, 32x32 RGB.


In [1]:
import os
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import platform
import tensorflow as tf
import scipy.io as sio

physical_gpus = tf.config.list_physical_devices("GPU")
if physical_gpus:
    for g in physical_gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print(e)
    print("Using GPU:", physical_gpus)
else:
    print("No GPU found.")
    if platform.system() == "Windows":
        print("On Windows, pip TensorFlow 2.11+ has no CUDA GPU. Use Colab GPU or WSL2+Linux.")
    else:
        print("Install NVIDIA drivers + CUDA TensorFlow per tensorflow.org/install/pip")

from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

with tf.device("/GPU:0" if physical_gpus else "/CPU:0"):
    _warm = tf.matmul(tf.random.normal([1024, 1024]), tf.random.normal([1024, 1024]))
print("Warmup tensor device:", _warm.device)



I0000 00:00:1776702165.543134     527 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1776702166.498771     527 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Warmup tensor device: /job:localhost/replica:0/task:0/device:GPU:0


## Load and preprocess SVHN

In [2]:
IMG_SHAPE = (32, 32, 3)
_gpu = bool(tf.config.list_physical_devices("GPU"))
BATCH_SIZE = 256 if _gpu else 128
EPOCHS = 10
noise_dim = 100
num_examples_to_generate = 16

train_url = "http://ufldl.stanford.edu/housenumbers/train_32x32.mat"
train_mat = keras.utils.get_file("train_32x32.mat", origin=train_url)
raw = sio.loadmat(train_mat)
x_train = np.transpose(raw["X"], (3, 0, 1, 2)).astype(np.float32)
y_train = raw["y"].reshape(-1).astype(np.int64)

y_train[y_train == 10] = 0

x_train = (x_train - 127.5) / 127.5
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(10000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Train examples:", x_train.shape[0])
print("Image shape:", x_train.shape[1:])
sample_batch = next(iter(train_dataset))[0]
print("Batch shape:", sample_batch.shape, "dtype:", sample_batch.dtype)


        0/182040794 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

    16384/182040794 ━━━━━━━━━━━━━━━━━━━━ 16:41 6us/step

    40960/182040794 ━━━━━━━━━━━━━━━━━━━━ 13:29 4us/step

    90112/182040794 ━━━━━━━━━━━━━━━━━━━━ 9:20 3us/step 

   147456/182040794 ━━━━━━━━━━━━━━━━━━━━ 7:27 2us/step

...

180772864/182040794 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

181927936/182040794 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

182040794/182040794 ━━━━━━━━━━━━━━━━━━━━ 64s 0us/step


Train examples: 73257
Image shape: (32, 32, 3)


Batch shape: (256, 32, 32, 3) dtype: <dtype: 'float32'>


### Sample real images

In [3]:
def grid_show(images, nrow=4, title=None):
    images = (images + 1.0) / 2.0
    images = np.clip(images.numpy(), 0.0, 1.0)
    fig, axes = plt.subplots(nrow, nrow, figsize=(8, 8))
    k = 0
    for r in range(nrow):
        for c in range(nrow):
            ax = axes[r, c]
            ax.imshow(images[k])
            ax.axis("off")
            k += 1
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


grid_show(sample_batch[:16], nrow=4, title="SVHN samples")


/tmp/ipykernel_527/805123452.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Generator and discriminator


In [4]:
def make_generator_model():
    model = keras.Sequential()
    model.add(layers.Dense(4 * 4 * 512, use_bias=False, input_shape=(noise_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Reshape((4, 4, 512)))

    model.add(
        layers.Conv2DTranspose(256, (4, 4), strides=(2, 2), padding="same", use_bias=False)
    )
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))

    model.add(
        layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding="same", use_bias=False)
    )
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Conv2DTranspose(64, (4, 4), strides=(2, 2), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))

    model.add(
        layers.Conv2DTranspose(
            3, (4, 4), strides=(1, 1), padding="same", use_bias=False, activation="tanh"
        )
    )
    return model


generator = make_generator_model()
generator.summary()


/home/dylan/dcgan_svhn_run/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8192)           │       819,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8192)           │        32,768 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 8, 8, 256)      │     2,097,152 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 16, 16, 128)    │       524,288 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 32, 32, 64)     │       131,072 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_3              │ (None, 32, 32, 3)      │         3,072 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,609,344 (13.77 MB)

 Trainable params: 3,592,064 (13.70 MB)

 Non-trainable params: 17,280 (67.50 KB)

In [5]:
noise = tf.random.normal([1, noise_dim])
generated_image = generator(noise, training=False)
plt.imshow(np.clip((generated_image[0].numpy() + 1.0) / 2.0, 0.0, 1.0))
plt.axis("off")
plt.title("Generator output (before training)")
plt.show()


/tmp/ipykernel_527/3958124298.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
def make_discriminator_model():
    model = keras.Sequential()
    model.add(
        layers.Conv2D(64, (4, 4), strides=(2, 2), padding="same", input_shape=IMG_SHAPE)
    )
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (4, 4), strides=(2, 2), padding="same"))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(256, (4, 4), strides=(2, 2), padding="same"))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(512, (4, 4), strides=(2, 2), padding="same"))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model


discriminator = make_discriminator_model()
discriminator.summary()


/home/dylan/dcgan_svhn_run/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 16, 16, 64)     │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 128)      │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_6 (LeakyReLU)       │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 2, 2, 512)      │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_7 (LeakyReLU)       │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,758,593 (10.52 MB)

 Trainable params: 2,758,593 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

## Losses and optimizers


In [7]:
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


generator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)


## Checkpoints


In [8]:
checkpoint_dir = "./training_checkpoints_svhn"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator,
)

seed = tf.random.normal([num_examples_to_generate, noise_dim])


## Training


In [9]:
@tf.function
def train_step(images_labels):
    images = images_labels[0]
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))
    return gen_loss, disc_loss


def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(np.clip((predictions[i, :, :, :].numpy() + 1.0) / 2.0, 0.0, 1.0))
        plt.axis("off")
    plt.savefig(f"image_at_epoch_{epoch:04d}.png")
    plt.show()
    plt.close(fig)


def train(dataset, epochs):
    for epoch in range(epochs):
        t0 = time.time()
        gen_losses, disc_losses = [], []
        for batch in dataset:
            gl, dl = train_step(batch)
            gen_losses.append(float(gl))
            disc_losses.append(float(dl))
        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"gen {float(np.mean(gen_losses)):.4f}, disc {float(np.mean(disc_losses)):.4f} - "
            f"{time.time() - t0:.1f}s"
        )
        generate_and_save_images(generator, epoch + 1, seed)
        if (epoch + 1) % 5 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)


In [10]:
train(train_dataset, EPOCHS)


E0000 00:00:1776702240.826518     527 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential_1_2/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


Epoch 1/10 - gen 2.5219, disc 0.8789 - 41.6s


/tmp/ipykernel_527/1539996459.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Epoch 2/10 - gen 1.4429, disc 0.8936 - 20.1s


Epoch 3/10 - gen 1.1169, disc 1.1115 - 19.9s


...

Epoch 6/10 - gen 0.8807, disc 1.2570 - 20.0s


Epoch 7/10 - gen 0.8241, disc 1.2970 - 20.6s


Epoch 8/10 - gen 0.7739, disc 1.3308 - 20.1s


Epoch 9/10 - gen 0.7956, disc 1.3105 - 20.6s


Epoch 10/10 - gen 0.8537, disc 1.2727 - 20.3s


## GIF


In [11]:
try:
    from PIL import Image

    anim_file = "dcgan_svhn.gif"
    frames = [Image.open(f"image_at_epoch_{e:04d}.png") for e in range(1, EPOCHS + 1)]
    frames[0].save(
        anim_file,
        save_all=True,
        append_images=frames[1:],
        duration=500,
        loop=0,
    )
    for f in frames:
        f.close()
    print("Wrote", anim_file)
except Exception as e:
    print(e)


Wrote dcgan_svhn.gif
